In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

import umap
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import train

import commons, models, utils, lightning_wrapper
from cough_datasets import CoughDatasets, CoughDatasetsCollate, CoughDiseaseBinaryBatchSampler

from s3prl.upstream.mockingjay.builder import PretrainedTransformer
from s3prl.upstream.mockingjay.model import TransformerSpecPredictionHead
import s3prl.optimizers
from IPython.display import Audio

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

In [ ]:
def _aprior(delta_hat: np.ndarray) -> np.ndarray:
    """ComBat EM prior: shape parameter for inverse gamma."""
    m = delta_hat.mean(axis=1)
    s2 = delta_hat.var(axis=1) + 1e-8
    return (2 * s2 + m**2) / s2


def _bprior(delta_hat: np.ndarray) -> np.ndarray:
    """ComBat EM prior: scale parameter for inverse gamma."""
    m = delta_hat.mean(axis=1)
    s2 = delta_hat.var(axis=1) + 1e-8
    return (m * s2 + m**3) / s2

def combat_harmonize(
    embeddings: np.ndarray,
    db_labels: np.ndarray,
    covariates: np.ndarray = None,
    n_iter: int = 30,
    conv_threshold: float = 1e-4,
) -> np.ndarray:
    """
    Empirical Bayes ComBat harmonization.
    
    Removes additive (gamma) and multiplicative (delta) batch effects
    while preserving biological signal via covariate design matrix.
    
    Parameters
    ----------
    embeddings   : (N, D) float array
    db_labels    : (N,) array of database IDs
    covariates   : (N, C) optional biological covariates to preserve
                   (e.g. disease_status encoded as 0/1)
    n_iter       : EM iterations for Bayesian shrinkage
    conv_threshold: convergence tolerance
    
    Returns
    -------
    harmonized   : (N, D) float array
    """
    N, D = embeddings.shape
    batches = np.unique(db_labels)
    n_batches = len(batches)
    
    # 1. Build design matrix
    # Intercept + covariates (biological signal to preserve)
    if covariates is not None:
        if covariates.ndim == 1:
            covariates = covariates.reshape(-1, 1)
        design = np.hstack([np.ones((N, 1)), covariates])
    else:
        design = np.ones((N, 1))
    
    # 2. Standardize overall
    grand_mean = embeddings.mean(axis=0)
    var_pooled = embeddings.var(axis=0) + 1e-8
    
    # 3. Regress out covariates to get residuals
    # Solve: embeddings = design @ B + residuals
    B, _, _, _ = np.linalg.lstsq(design, embeddings, rcond=None)
    residuals = embeddings - design @ B
    
    # 4. Standardize residuals
    stand_mean = (grand_mean / np.sqrt(var_pooled))
    s_data = (embeddings - grand_mean) / np.sqrt(var_pooled)
    
    # 5. Estimate batch effects (gamma_hat = additive, delta_hat = multiplicative)
    gamma_hat = np.zeros((n_batches, D))
    delta_hat = np.zeros((n_batches, D))
    
    batch_idx = {}
    for i, b in enumerate(batches):
        mask = db_labels == b
        batch_idx[i] = mask
        batch_data = s_data[mask]
        gamma_hat[i] = batch_data.mean(axis=0)
        delta_hat[i] = batch_data.var(axis=0) + 1e-8
    
    # 6. Empirical Bayes priors
    gamma_bar = gamma_hat.mean(axis=0)
    t2 = gamma_hat.var(axis=0) + 1e-8
    
    a_prior = _aprior(delta_hat)
    b_prior = _bprior(delta_hat)
    
    # 7. EM iteration for posterior estimates
    gamma_star = gamma_hat.copy()
    delta_star = delta_hat.copy()
    
    for _ in range(n_iter):
        gamma_star_new = np.zeros_like(gamma_hat)
        delta_star_new = np.zeros_like(delta_hat)
        
        for i, b in enumerate(batches):
            mask = batch_idx[i]
            n_i = mask.sum()
            batch_data = s_data[mask]
            
            # Posterior gamma (additive)
            gamma_star_new[i] = (
                (t2 * n_i * batch_data.mean(axis=0) + delta_star[i] * gamma_bar)
                / (t2 * n_i + delta_star[i])
            )
            
            # Posterior delta (multiplicative) via inverse gamma
            sum_sq = ((batch_data - gamma_star_new[i]) ** 2).sum(axis=0)
            delta_star_new[i] = (b_prior[i] + 0.5 * sum_sq) / (a_prior[i] + n_i / 2.0 - 1)
            delta_star_new[i] = np.maximum(delta_star_new[i], 1e-8)
        
        # Check convergence
        g_change = np.abs(gamma_star_new - gamma_star).max()
        d_change = np.abs(delta_star_new - delta_star).max()
        gamma_star = gamma_star_new
        delta_star = delta_star_new
        
        if g_change < conv_threshold and d_change < conv_threshold:
            break
    
    # 8. Apply correction
    harmonized = s_data.copy()
    for i, b in enumerate(batches):
        mask = batch_idx[i]
        harmonized[mask] = (
            (s_data[mask] - gamma_star[i]) / np.sqrt(delta_star[i])
        )
    
    # 9. Rescale back to original space
    harmonized = harmonized * np.sqrt(var_pooled) + grand_mean
    
    # Re-apply biological covariates
    harmonized = harmonized + design @ B - grand_mean
    
    return harmonized

def compute_mmd(X: np.ndarray, Y: np.ndarray, gamma: float = None) -> float:
    """
    Unbiased MMD² with RBF kernel.
    Lower = more similar distributions (better harmonization).
    """
    if gamma is None:
        gamma = 1.0 / X.shape[1]
    
    def rbf_kernel(A, B):
        dists = np.sum((A[:, None] - B[None, :]) ** 2, axis=2)
        return np.exp(-gamma * dists)
    
    Kxx = rbf_kernel(X, X)
    Kyy = rbf_kernel(Y, Y)
    Kxy = rbf_kernel(X, Y)
    
    n, m = len(X), len(Y)
    mmd2 = (
        (Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
        + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
        - 2 * Kxy.mean()
    )
    return float(mmd2)

def pairwise_mmd_matrix(embeddings: np.ndarray, db_labels: np.ndarray) -> pd.DataFrame:
    """Compute pairwise MMD between all database pairs."""
    batches = np.unique(db_labels)
    n = len(batches)
    matrix = np.zeros((n, n))
    
    for i, b1 in enumerate(batches):
        for j, b2 in enumerate(batches):
            if i != j:
                X = embeddings[db_labels == b1]
                Y = embeddings[db_labels == b2]
                # Subsample for speed if large
                if len(X) > 200: X = X[np.random.choice(len(X), 200, replace=False)]
                if len(Y) > 200: Y = Y[np.random.choice(len(Y), 200, replace=False)]
                matrix[i, j] = compute_mmd(X, Y)
    
    return pd.DataFrame(matrix, index=batches, columns=batches)

def get_classifiers() -> dict:
    """
    Return a dict of classifiers to compare.
    All wrapped in a Pipeline with StandardScaler + PCA for fair comparison.
    PCA(50) reduces 768-dim embeddings to a stable space for all classifiers.
    """
    pca_step = ("pca", PCA(n_components=50, random_state=42))
    scale_step = ("scaler", StandardScaler())

    return {
        "Logistic Regression": Pipeline([
            scale_step, pca_step,
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]),
        "SVM (RBF)": Pipeline([
            scale_step, pca_step,
            ("clf", SVC(kernel="rbf", probability=True, C=1.0, random_state=42)),
        ]),
        "Random Forest": Pipeline([
            scale_step, pca_step,
            ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ]),
        "Gradient Boosting": Pipeline([
            scale_step, pca_step,
            ("clf", GradientBoostingClassifier(n_estimators=200, random_state=42)),
        ]),
    }

In [ ]:
import sys
original_optimizer = sys.modules.get("optimizers")
sys.modules["optimizers"] = s3prl.optimizers

class TERA_TryDownstream(nn.Module):
    def __init__(self, input_size, **kwargs):
        super(TERA_TryDownstream, self).__init__()

        options = {
            "load_pretrain": "True",
            "no_grad": "True",
            "dropout": "default",
            "spec_aug": "False",
            "spec_aug_prev": "False",
            "output_hidden_states": "True",
            "permute_input": "False",
        }
        options["ckpt_file"] = "/run/media/fourier/Data1/Pras/Thesis_Nexus/s3prl/s3prl/result/pretrain/tera_cough_ssldata_lowlr/states-990000.ckpt"
        options["select_layer"] = -1
        
        pretrained_dict = torch.load(options["ckpt_file"], weights_only=False)
        transformer_state = pretrained_dict['Transformer']
        spechead_state = pretrained_dict['SpecHead']

        self.tera_model = PretrainedTransformer(options, inp_dim=-1)
        self.tera_model.model.load_state_dict(transformer_state, strict=True)
        self.tera_model.eval()

        self.spechead_model = TransformerSpecPredictionHead(self.tera_model.model_config, self.tera_model.spec_dim)
        self.spechead_model.load_state_dict(spechead_state, strict=True)
        self.spechead_model.eval()

        dropout = 0.1
        embed_dim = 768 * 2

    def forward(self, x, attention_mask=None, grl_lambda=0.0):
        x = x.squeeze(1)
        with torch.no_grad():
            x = self.tera_model(x)[0] # Index 0 = Last Hidden, Index 1 All Transformwer
            x = torch.nan_to_num(x, nan=0.0)
            reconstructed_mel, _ = self.spechead_model(x)
            reconstructed_mel = reconstructed_mel.transpose(1, 2)

        feature_embedding = x.mean(dim=1)
        
        return {
            "embeddings": feature_embedding,
            "reconstructed_mel": reconstructed_mel,
        }
    
model = TERA_TryDownstream(1)
model.cuda()
model.eval()

del sys.modules["optimizers"]

In [ ]:
class Qwen3_Extractor(nn.Module):
    def __init__(self, dummy, **kwargs):
        super(Qwen3_Extractor, self).__init__()

        from peft import get_peft_model, LoraConfig, TaskType
        from transformers import Qwen3OmniMoeThinkerForConditionalGeneration

        temp_model = Qwen3OmniMoeThinkerForConditionalGeneration.from_pretrained(
            "/run/media/fourier/Data1/Pras/pretrain_models/Qwen3-Omni-30B-A3B-Thinking",
            torch_dtype="auto",
            device_map="cpu"
        )
        self.audio_tower = temp_model.audio_tower
        self.audio_tower.cuda()
        del temp_model
        import gc
        import torch
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        self.audiotower_hidden_dim = self.audio_tower.config.output_dim

    def after_cnn_len(self, L):
        L = (L - 1) // 2 + 1
        L = (L - 1) // 2 + 1
        L = (L - 1) // 2 + 1
        return L

    def forward(self, input_features, attention_mask=None, **kwargs):
        input_features = input_features.to(torch.bfloat16)
        feature_attention_mask = attention_mask.long()
        if feature_attention_mask is not None:
            audio_feature_lengths = torch.sum(feature_attention_mask, dim=1)
            input_features = input_features.permute(
                0, 2, 1)[feature_attention_mask.bool()].permute(1, 0)
        else:
            audio_feature_lengths = None

        feature_lens = audio_feature_lengths if audio_feature_lengths is not None else feature_attention_mask.sum(-1)
        audio_outputs = self.audio_tower(
            input_features,
            feature_lens=feature_lens,
        )
        audio_features = audio_outputs.last_hidden_state
        post_lens = torch.tensor(
            [self.after_cnn_len(l.item()) for l in feature_lens],
            device=feature_lens.device
        )

        total = audio_features.size(0)
        delta = total - post_lens.sum()
        if delta != 0:
            post_lens[-1] += delta

        audio_features = audio_features.split(post_lens.tolist(), dim=0)
        audio_features = pad_sequence(audio_features, batch_first=True) # for Attentive Pooling
        audio_features = audio_features.to(torch.float32)

        mean = audio_features.mean(dim=1)
        std = audio_features.std(dim=1)

        embeddings = torch.cat([mean, std], dim=1)
        return {
            "embeddings": embeddings,
        }

model = Qwen3_Extractor(1)
model.cuda()
model.eval()

In [ ]:
parser = train.parse_args()
args = parser.parse_args(["--init", "--model_name", "dev", "--pooling_model",
                          "qwen", "--config_path", "configs/general.json"]) # qwen

model_dir = os.path.join("./logs", args.model_name)
os.makedirs(model_dir, exist_ok=True)
port = None

config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
hps = train.load_config(config_path, model_dir, args)

df_train, _ = train.load_data(hps)
collate_fn = train.get_collate_fn(hps)
target_labels = df_train[hps.data.target_column]

_, coda_loader = train.prepare_fold_data(
    df_train, df_train, hps, collate_fn,
    use_precomputed=args.use_precomputed,
    precomputed_dir=args.precomputed_dir
)

df_cirdz = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata_cirdz.csv.train')
df_cirdz = df_cirdz.reset_index(drop=True)
df_cirdz = df_cirdz[hps.data.column_order]

_, cirdz_loader = train.prepare_fold_data(
    df_cirdz, df_cirdz, hps, collate_fn,
    use_precomputed=args.use_precomputed,
    precomputed_dir=args.precomputed_dir
)

In [ ]:
train_embeds = []
train_wavs = []
with torch.no_grad():
    for idx, batch in tqdm(enumerate(coda_loader), total=len(coda_loader)):
        wavs_names, audio, _, attention_masks, dse_ids, [patient_ids, _, tabular_ids, _]  = batch
        audio = audio.cuda()
        out_model = model(audio, attention_mask=attention_masks)
        embed = out_model['embeddings']

        dse_ids = torch.argmax(dse_ids, dim=1)
        train_wavs.extend(wavs_names)
        train_embeds.append(embed.cpu())

train_wavs = np.array(train_wavs)
train_embeds = torch.cat(train_embeds, dim=0).numpy()

df_train = df_train.set_index("path_file").loc[train_wavs].reset_index()
df_train["embed"] = list(train_embeds)
df_train["db"] = 0

train_embeds = []
train_wavs = []
with torch.no_grad():
    for idx, batch in tqdm(enumerate(cirdz_loader), total=len(cirdz_loader)):
        wavs_names, audio, _, attention_masks, dse_ids, [patient_ids, _, tabular_ids, _]  = batch
        audio = audio.cuda()
        out_model = model(audio, attention_mask=attention_masks)
        embed = out_model['embeddings']

        dse_ids = torch.argmax(dse_ids, dim=1)
        train_wavs.extend(wavs_names)
        train_embeds.append(embed.cpu())

train_wavs = np.array(train_wavs)
train_embeds = torch.cat(train_embeds, dim=0).numpy()

df_cirdz = df_cirdz.set_index("path_file").loc[train_wavs].reset_index()
df_cirdz["embed"] = list(train_embeds)
df_cirdz["db"] = 1

df_combine = pd.concat([df_train, df_cirdz]).reset_index(drop=True)

In [ ]:
raw_emb = np.stack(df_combine['embed'].values)
db_labels = df_combine['db'].values
disease_labels = df_combine['disease_status'].values

N, D = raw_emb.shape
print(f"  Embeddings: {N} samples × {D} dims")
print(f"  Databases : {dict(zip(*np.unique(db_labels, return_counts=True)))}")
print(f"  Disease   : {dict(zip(*np.unique(disease_labels, return_counts=True)))}")

In [ ]:
df_filtered = df_combine #[df_combine["disease_status_rev"] != 2].copy().reset_index(drop=True)
df_filtered = df_filtered[df_filtered['db'] == 0]
X = np.vstack(df_filtered["embed"].values).astype(np.float32)
y = df_filtered["disease_status"].astype(int).values
participant = df_filtered["participant"].values
db_labels = df_filtered["db"].values

unique_dbs = np.unique(db_labels)
print(f"\n  Class balance: Acoustic-TB={y.sum()}  Non-TB={(y==0).sum()}")
print(f"  Databases: {dict(zip(*np.unique(db_labels, return_counts=True)))}")
print(f"  Embedding dim: {X.shape[1]}")

In [ ]:
print(f"Scenario A")
classifiers = get_classifiers()
results = {}

cv = GroupKFold(n_splits=5)
splits = list(cv.split(X, y, groups=participant))

for clf_name, clf in classifiers.items():
    print(f"\n  [{clf_name}]")
    fold_aurocs, fold_auprcs = [], []
    all_y_true, all_y_score = [], []
    fold_roc_curves = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_score = clf.predict_proba(X_test)[:, 1]

        auroc = roc_auc_score(y_test, y_score)
        auprc = average_precision_score(y_test, y_score)
        fpr, tpr, _ = roc_curve(y_test, y_score)

        fold_aurocs.append(auroc)
        fold_auprcs.append(auprc)
        all_y_true.extend(y_test)
        all_y_score.extend(y_score)
        fold_roc_curves.append((fpr, tpr))

    mean_auroc = np.mean(fold_aurocs)
    std_auroc  = np.std(fold_aurocs)
    mean_auprc = np.mean(fold_auprcs)

    print(f"    → Mean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f"    → Mean AUPRC: {mean_auprc:.4f}")

    results[clf_name] = {
        "fold_aurocs":      fold_aurocs,
        "mean_auroc":       mean_auroc,
        "std_auroc":        std_auroc,
        "fold_auprcs":      fold_auprcs,
        "mean_auprc":       mean_auprc,
        "all_y_true":       np.array(all_y_true),
        "all_y_score":      np.array(all_y_score),
        "fold_roc_curves":  fold_roc_curves,
    }

## COMBAT

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA

In [ ]:
covariates = disease_labels.astype(float).reshape(-1, 1)
combat_emb = combat_harmonize(raw_emb, db_labels, covariates=covariates)
embeddings = {"raw": raw_emb, "ComBat": combat_emb}

mmd_raw = pairwise_mmd_matrix(raw_emb, db_labels)
mmd_combat = pairwise_mmd_matrix(combat_emb, db_labels)
print({
    "Raw":     mmd_raw.values[mmd_raw.values > 0].mean(),
    "ComBat":  mmd_combat.values[mmd_combat.values > 0].mean(),
})

r2_result = {}
for name, emb in embeddings.items():
    pca = PCA(n_components=50, random_state=42)
    pcs = pca.fit_transform(emb)

    r2_db = Ridge().fit(db_labels.reshape(-1, 1), pcs).score(db_labels.reshape(-1, 1), pcs)
    r2_dis = Ridge().fit(disease_labels.reshape(-1, 1), pcs).score(disease_labels.reshape(-1, 1), pcs)
    r2_result[name] = {
        "r2_db": r2_db,
        "r2_dis": r2_dis,
    }
r2_result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Embedding Space Before & After Harmonization")

for col, (name, emb) in enumerate(embeddings.items()):
    pca = PCA(n_components=2, random_state=42)
    pcs = pca.fit_transform(emb)
    var1, var2 = pca.explained_variance_ratio_ * 100

    ax = axes[col]
    unique_dbs = np.unique(db_labels)
    for db in unique_dbs:
        mask = db_labels == db
        ax.scatter(pcs[mask, 0], pcs[mask, 1], label=str(db), alpha=0.6, s=18, linewidths=0)
    ax.set_title(f"{name}", fontsize=13, fontweight="bold")
    ax.set_xlabel(f"PC1 ({var1:.1f}%)", fontsize=9)
    ax.set_ylabel(f"PC2 ({var2:.1f}%)", fontsize=9)
    ax.legend(fontsize=7, markerscale=1.5, title="DB", title_fontsize=8)
    ax.tick_params(labelsize=8)

In [ ]:
df_combine["embed"] = list(combat_emb)

In [ ]:
df_filtered = df_combine #[df_combine["disease_status_rev"] != 2].copy().reset_index(drop=True)
df_filtered = df_filtered[df_filtered['db'] == 0]
X = np.vstack(df_filtered["embed"].values).astype(np.float32)
y = df_filtered["disease_status"].astype(int).values
participant = df_filtered["participant"].values
db_labels = df_filtered["db"].values

unique_dbs = np.unique(db_labels)
print(f"\n  Class balance: Acoustic-TB={y.sum()}  Non-TB={(y==0).sum()}")
print(f"  Databases: {dict(zip(*np.unique(db_labels, return_counts=True)))}")
print(f"  Embedding dim: {X.shape[1]}")

print(f"Scenario A")
classifiers = get_classifiers()
results = {}

cv = GroupKFold(n_splits=5)
splits = list(cv.split(X, y, groups=participant))

for clf_name, clf in classifiers.items():
    print(f"\n  [{clf_name}]")
    fold_aurocs, fold_auprcs = [], []
    all_y_true, all_y_score = [], []
    fold_roc_curves = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_score = clf.predict_proba(X_test)[:, 1]

        auroc = roc_auc_score(y_test, y_score)
        auprc = average_precision_score(y_test, y_score)
        fpr, tpr, _ = roc_curve(y_test, y_score)

        fold_aurocs.append(auroc)
        fold_auprcs.append(auprc)
        all_y_true.extend(y_test)
        all_y_score.extend(y_score)
        fold_roc_curves.append((fpr, tpr))

        print(f"    Fold {fold_idx+1}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}  "
                f"(test n={len(y_test)}, TB={y_test.sum()})")

    mean_auroc = np.mean(fold_aurocs)
    std_auroc  = np.std(fold_aurocs)
    mean_auprc = np.mean(fold_auprcs)

    print(f"    → Mean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f"    → Mean AUPRC: {mean_auprc:.4f}")

    results[clf_name] = {
        "fold_aurocs":      fold_aurocs,
        "mean_auroc":       mean_auroc,
        "std_auroc":        std_auroc,
        "fold_auprcs":      fold_auprcs,
        "mean_auprc":       mean_auprc,
        "all_y_true":       np.array(all_y_true),
        "all_y_score":      np.array(all_y_score),
        "fold_roc_curves":  fold_roc_curves,
    }
    break

: 

## Outlier

In [ ]:
raw_emb = np.stack(df_combine['embed'].values)
db_labels = df_combine['db'].values
disease_labels = df_combine['disease_status'].values

In [ ]:
contamination = 0.05
pca = PCA(n_components=min(50, raw_emb.shape[1]), random_state=42)
emb_pca = pca.fit_transform(raw_emb)

# ─────────────────────────────────────────────
# METHOD A: Isolation Forest
# ─────────────────────────────────────────────
clf = IsolationForest(
    n_estimators=200,
    contamination=contamination, # Expected fraction of outliers 5%
    random_state=42,
    n_jobs=-1,
)
clf.fit_predict(emb_pca)        # 1 = inlier, -1 = outlier
if_scores = clf.score_samples(emb_pca)      # lower = more anomalous
if_scores = -if_scores  # IF scores are negative; negate so high = bad
if_scores = (if_scores - if_scores.min()) / (if_scores.max() - if_scores.min() + 1e-8)

# ─────────────────────────────────────────────
# METHOD B: Per-Class Centroid Distance
# ─────────────────────────────────────────────
threshold_sigma = 2.5 # threshold_sigma  : SD multiplier for outlier boundary
distances = np.zeros(len(raw_emb))
unique_classes = np.unique(db_labels)

for cls in unique_classes:
    mask = db_labels == cls
    subset = emb_pca[mask]
    centroid = subset.mean(axis=0)
    
    # Euclidean distance from centroid
    dists = np.linalg.norm(subset - centroid, axis=1)
    
    # Normalize by median (robust to outliers in the distance distribution)
    median_dist = np.median(dists)
    mad = np.median(np.abs(dists - median_dist)) + 1e-8  # Median Absolute Deviation
    normalized = (dists - median_dist) / mad
    distances[mask] = normalized # sigma

cd_norm = np.clip(distances, 0, None)  # remove negative (inlier side)
cd_norm = (cd_norm - cd_norm.min()) / (cd_norm.max() - cd_norm.min() + 1e-8)

# Ensemble
ensemble_scores = 0.55 * if_scores + 0.45 * cd_norm
threshold = np.percentile(ensemble_scores, 100 * (1 - contamination))
outlier_labels = np.where(ensemble_scores >= threshold, -1, 1)

outlier_mask = outlier_labels == -1
inlier_mask = outlier_labels == 1
n_outliers = outlier_mask.sum()
print(f"  outliers: {n_outliers} ({outlier_mask.mean()*100:.1f}%)")

In [ ]:
pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(raw_emb)
var1, var2 = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Embedding Space Before & After Cleaning")

ax = axes[0]
ax.scatter(pcs[:, 0], pcs[:, 1], alpha=0.6, s=18, linewidths=0)
ax.set_title(f"Uncleaned", fontsize=13, fontweight="bold")
ax.set_xlabel(f"PC1 ({var1:.1f}%)", fontsize=9)
ax.set_ylabel(f"PC2 ({var2:.1f}%)", fontsize=9)
ax.tick_params(labelsize=8)

ax = axes[1]
ax.scatter(pcs[inlier_mask, 0], pcs[inlier_mask, 1], alpha=0.6, s=18, linewidths=0)
ax.set_title(f"Uncleaned", fontsize=13, fontweight="bold")
ax.set_xlabel(f"PC1 ({var1:.1f}%)", fontsize=9)
ax.set_ylabel(f"PC2 ({var2:.1f}%)", fontsize=9)
ax.tick_params(labelsize=8)

In [ ]:
df_combine = df_combine[~outlier_mask].reset_index(drop=True)

## Intra Classs

In [ ]:
from sklearn.neighbors import KernelDensity
from sklearn.neighbors import NearestNeighbors

def _sigmoid_normalize(scores: np.ndarray) -> np.ndarray:
    """Center and sigmoid-normalize scores to [0, 1]."""
    centered = (scores - scores.mean()) / (scores.std() + 1e-8)
    return 1.0 / (1.0 + np.exp(-centered))

def _minmax_normalize(scores: np.ndarray) -> np.ndarray:
    lo, hi = scores.min(), scores.max()
    return (scores - lo) / (hi - lo + 1e-8)

def best_gmm(data, max_k=6, random_state=42):
    best_bic, best_k = np.inf, 1
    max_k = min(max_k, len(data) // 20)  # need enough samples per component
    max_k = max(max_k, 1)
    for k in range(1, max_k + 1):
        try:
            g = GaussianMixture(n_components=k, covariance_type="diag",
                                random_state=random_state, n_init=3)
            g.fit(data)
            bic = g.bic(data)
            if bic < best_bic:
                best_bic, best_k = bic, k
        except Exception:
            break
    gmm = GaussianMixture(n_components=best_k, covariance_type="diag",
                            random_state=random_state, n_init=5)
    gmm.fit(data)
    print(f"    GMM components selected (BIC): {best_k}")
    return gmm

def find_optimal_k(
    embeddings: np.ndarray,
    k_range: range = None,
    method: str = "silhouette",
    random_state: int = 42,
) -> tuple[int, pd.DataFrame]:
    """
    Find optimal number of clusters via silhouette + elbow analysis.
    
    Returns
    -------
    best_k     : recommended number of clusters
    metrics_df : per-k metrics table
    """
    if k_range is None:
        k_range = range(2, min(8, len(embeddings) // 10 + 1))
    
    pca = PCA(n_components=min(30, embeddings.shape[1]), random_state=random_state)
    emb_pca = pca.fit_transform(embeddings)
    
    records = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        labels = km.fit_predict(emb_pca)
        
        sil = silhouette_score(emb_pca, labels)
        inertia = km.inertia_
        db = davies_bouldin_score(emb_pca, labels)
        records.append({"k": k, "silhouette": sil, "inertia": inertia, "davies_bouldin": db})
    
    df = pd.DataFrame(records)
    
    # Best k = highest silhouette (or use elbow)
    best_k = int(df.loc[df["silhouette"].idxmax(), "k"])
    
    return best_k, df

def cluster_within_class(
    embeddings: np.ndarray,
    class_labels: np.ndarray,
    target_class,
    n_clusters: int = None,
    method: str = "kmeans",
    random_state: int = 42,
) -> tuple[np.ndarray, KMeans, pd.DataFrame]:
    """
    Cluster samples within a specific class (e.g., TB only).
    
    Returns
    -------
    cluster_labels : (N_class,) cluster assignments
    model          : fitted clustering model
    k_metrics      : cluster quality metrics
    """
    mask = class_labels == target_class
    subset = embeddings[mask]
    
    pca = PCA(n_components=min(30, subset.shape[1]), random_state=random_state)
    subset_pca = pca.fit_transform(subset)
    
    if n_clusters is None:
        print(f"    Auto-selecting k for class '{target_class}'...")
        n_clusters, k_metrics = find_optimal_k(subset, random_state=random_state)
        print(f"    → Optimal k = {n_clusters}")
    else:
        k_metrics = pd.DataFrame()
    
    if method == "kmeans":
        model = KMeans(n_clusters=n_clusters, n_init=20, random_state=random_state)
        cluster_labels = model.fit_predict(subset_pca)
    elif method == "gmm":
        model = GaussianMixture(n_components=n_clusters, random_state=random_state,
                                covariance_type="full", n_init=5)
        cluster_labels = model.fit_predict(subset_pca)
    
    return cluster_labels, model, k_metrics, pca



In [ ]:
raw_emb = np.stack(df_combine['embed'].values)
db_labels = df_combine['db'].values
disease_labels = df_combine['disease_status'].values

tb_mask = disease_labels == 1

In [ ]:
all_scores = {}
pca_components = 30
k_knn = 15

pca = PCA(n_components=pca_components, random_state=42)
emb_pca = pca.fit_transform(raw_emb)

tb_emb   = emb_pca[tb_mask]
nontb_emb = emb_pca[~tb_mask]

In [ ]:
"""
One-Class Kernel Density Estimation on Non-TB samples.

Intuition
---------
Model the "normal cough" (Non-TB) distribution using KDE.
A sample that falls in a LOW-density region of Non-TB space
is acoustically unusual → likely TB.

Score = 1 - P(x | Non-TB)  →  high score = far from Non-TB = likely TB

This is purely unsupervised with respect to TB labels:
the TB label is only used to define what "normal" means (Non-TB).
No linearity assumption. Works on the full non-linear manifold
via the kernel.

Parameters
----------
pca_components : dims to reduce to before KDE (KDE suffers in very high dims)
"""

# Bandwidth selection via Scott's rule
bandwidth = nontb_emb.shape[0] ** (-1.0 / (nontb_emb.shape[1] + 4))
bandwidth = max(bandwidth, 0.1)

kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth)
kde.fit(nontb_emb)

log_density = kde.score_samples(emb_pca)  # (N,) — higher = more like Non-TB
scores = -log_density  # Invert: high score = anomalous w.r.t. Non-TB = likely TB
all_scores["kde"] = _sigmoid_normalize(scores)

In [ ]:
"""
k-Nearest Neighbor distance-based acoustic TB likelihood.

Intuition
---------
For each sample, compute:
    d_TB     = mean distance to its k nearest TB neighbors
    d_NonTB  = mean distance to its k nearest Non-TB neighbors

Score = d_NonTB / (d_TB + d_NonTB)
    → high score = closer to TB neighborhood = likely TB
    → low score  = closer to Non-TB neighborhood = likely Non-TB

Makes NO distributional assumptions. Captures non-linear, multi-modal
cluster structure. Each sample is judged by its local neighborhood,
not a global decision boundary.

Parameters
----------
k : number of neighbors (15–25 works well for large datasets)
"""
k_tb, _ = find_optimal_k(tb_emb, random_state=42)
k_nontb, _ = find_optimal_k(nontb_emb, random_state=42)
# k_tb    = min(k_knn, len(tb_emb))
# k_nontb = min(k_knn, len(nontb_emb))

# Fit separate NN indices for each class
nn_tb = NearestNeighbors(n_neighbors=k_tb, metric="euclidean", n_jobs=-1)
nn_tb.fit(tb_emb)

nn_nontb = NearestNeighbors(n_neighbors=k_nontb, metric="euclidean", n_jobs=-1)
nn_nontb.fit(nontb_emb)

# Query distances for all samples
d_tb,    _ = nn_tb.kneighbors(emb_pca)     # (N, k_tb)
d_nontb, _ = nn_nontb.kneighbors(emb_pca)  # (N, k_nontb)

mean_d_tb    = d_tb.mean(axis=1)    # (N,)
mean_d_nontb = d_nontb.mean(axis=1) # (N,)

# Total Distance Score, [nonTB] ----------- x --- [TB] -> 0 - 1, tell how far from non tb
scores = mean_d_nontb / (mean_d_tb + mean_d_nontb + 1e-8) # Relative proximity score: high = closer to TB
all_scores["knn"]  = _minmax_normalize(scores)

In [ ]:
"""
Gaussian Mixture Model posterior: P(TB | x).

Intuition
---------
Fit a separate GMM on TB samples and on Non-TB samples.
Each GMM models the multi-modal, non-linear structure of its class.
Score = P(x | GMM_TB) / (P(x | GMM_TB) + P(x | GMM_NonTB))
        = Bayesian posterior P(TB | x) assuming equal class priors.

Unlike LDA, this:
    - Handles multi-modal distributions (multiple acoustic TB subtypes)
    - Captures non-linear cluster shapes via mixture of Gaussians
    - Gives a proper probabilistic score with Bayesian interpretation

n_components auto-selected via BIC if not provided.

Parameters
----------
n_components_tb    : GMM components for TB class (None = auto via BIC)
n_components_nontb : GMM components for Non-TB class (None = auto via BIC)
"""

gmm_tb = best_gmm(tb_emb, random_state=42)
gmm_nontb = best_gmm(nontb_emb, random_state=42)

# Log-likelihoods
log_p_tb    = gmm_tb.score_samples(emb_pca)    # (N,) log P(x | TB)
log_p_nontb = gmm_nontb.score_samples(emb_pca) # (N,) log P(x | Non-TB)

# Stable log-sum-exp for posterior
# P(TB|x) = exp(log_p_tb) / (exp(log_p_tb) + exp(log_p_nontb)) , The probability that the sample has TB given the observed data x
log_max = np.maximum(log_p_tb, log_p_nontb)
posterior_tb = np.exp(log_p_tb - log_max) / (
    np.exp(log_p_tb - log_max) + np.exp(log_p_nontb - log_max) + 1e-8
)
all_scores["gmm"] = posterior_tb.astype(np.float32)

In [ ]:
# Per-method breakdown
for m, s in all_scores.items():
    if m == "ensemble": continue
    print(f"  [{m.upper()}] TB: {s[tb_mask].mean():.3f} ± {s[tb_mask].std():.3f} | " f"Non-TB: {s[~tb_mask].mean():.3f} ± {s[~tb_mask].std():.3f}")

In [ ]:
# Simple average — each method captures different aspects
scores_ensembles = np.mean(list(all_scores.values()), axis=0)
all_scores["ensemble"] = scores_ensembles
acoustic_scores = np.mean([all_scores["gmm"], all_scores["knn"]], axis=0)

print(f"  Ensemble TB mean score:     {acoustic_scores[tb_mask].mean():.3f} ± {acoustic_scores[tb_mask].std():.3f}")
print(f"  Ensemble Non-TB mean score: {acoustic_scores[~tb_mask].mean():.3f} ± {acoustic_scores[~tb_mask].std():.3f}")

In [ ]:
from scipy.stats import mannwhitneyu

methods = [m for m in ["knn", "gmm", "ensemble"] if m in all_scores]
n = len(methods)

fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
fig.suptitle("Acoustic TB Likelihood — Method Comparison", fontsize=14, fontweight="bold", y=1.01)

method_labels = {"kde": "One-Class KDE\n(Non-TB density anomaly)",
                     "knn": "k-NN Distance\n(local neighborhood)",
                     "gmm": "GMM Posterior\nP(TB|x)",
                     "ensemble": "Ensemble\n(average of 3)"}
colors = {"TB": "#C0392B", "Non-TB": "#2980B9"}

for col, method in enumerate(methods):
    scores = all_scores[method]
    tb_scores    = scores[tb_mask]
    nontb_scores = scores[~tb_mask]

    # Row 0: distribution
    ax = axes[0, col]
    bins = np.linspace(0, 1, 35)
    ax.hist(nontb_scores, bins=bins, color=colors["Non-TB"], alpha=0.65,
            label=f"Non-TB (μ={nontb_scores.mean():.2f})", density=True)
    ax.hist(tb_scores, bins=bins, color=colors["TB"], alpha=0.65,
            label=f"TB (μ={tb_scores.mean():.2f})", density=True)

    # AUC-like separation: how well does this score separate classes?
    from sklearn.metrics import roc_auc_score
    try:
        auc = roc_auc_score(tb_mask.astype(int), scores)
        auc = max(auc, 1 - auc)  # always report > 0.5
    except Exception:
        auc = float("nan")

    ax.set_title(f"{method_labels.get(method, method)}\nAUROC = {auc:.3f}",
                    fontsize=10, fontweight="bold")
    ax.set_xlabel("Score", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.legend(fontsize=8)

    # Row 1: box plot per class
    ax = axes[1, col]
    data_plot = [nontb_scores, tb_scores]
    bp = ax.boxplot(data_plot, patch_artist=True, widths=0.5,
                    medianprops=dict(color="white", linewidth=2))
    for patch, color in zip(bp["boxes"], [colors["Non-TB"], colors["TB"]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Non-TB", "TB"], fontsize=10)
    ax.set_ylabel("Score", fontsize=9)

    # Statistical test
    stat, pval = mannwhitneyu(tb_scores, nontb_scores, alternative="two-sided")
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
    ymax = max(tb_scores.max(), nontb_scores.max())
    ax.annotate("", xy=(2, ymax * 1.05), xytext=(1, ymax * 1.05),
                arrowprops=dict(arrowstyle="-", color="black"))
    ax.text(1.5, ymax * 1.07, f"p={pval:.2e} {sig}",
            ha="center", va="bottom", fontsize=8)

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
# ─────────────────────────────────────────────
# CLUSTERING WITHIN EACH CLASS
# ─────────────────────────────────────────────
pca_components = 30
tb_embeddings = raw_emb[tb_mask]
n_clusters, _ = find_optimal_k(tb_embeddings, random_state=42)

pca = PCA(n_components=pca_components, random_state=42)
tb_embeddings_pca = pca.fit_transform(tb_embeddings)

model_knn_tb = KMeans(n_clusters=n_clusters, n_init=20, random_state=42)
tb_clusters = model_knn_tb.fit_predict(tb_embeddings_pca)

sil_tb = silhouette_score(pca.transform(tb_embeddings), tb_clusters) if len(np.unique(tb_clusters)) > 1 else 0
print(f"  TB clusters: {len(np.unique(tb_clusters))} | Silhouette: {sil_tb:.3f}")

In [ ]:
df_tb = df_combine[tb_mask].copy().reset_index(drop=True)
tb_acoustic_scores = acoustic_scores[tb_mask]
ntb_acoustic_scores = acoustic_scores[~tb_mask]

df_tb["cluster"] = tb_clusters
df_tb["acoustic_tb_score"] = tb_acoustic_scores

threshold = np.median(tb_acoustic_scores) # Try Median
threshold = np.percentile(ntb_acoustic_scores, 100) + 0.05
df_tb["tb_acoustic_subtype"] = np.where(
    tb_acoustic_scores >= threshold,
    "Acoustic-TB",
    "Label-only-TB"
)

# Per-cluster: determine if cluster is predominantly Acoustic or Label-only , WHat Fr?????
cluster_means = df_tb.groupby("cluster")["acoustic_tb_score"].mean()
df_tb["cluster_type"] = df_tb["cluster"].map(
    lambda c: "Acoustic-TB Cluster" if cluster_means[c] >= threshold else "Label-only-TB Cluster"
)

n_acoustic = (df_tb["tb_acoustic_subtype"] == "Acoustic-TB").sum()
n_labelonly = (df_tb["tb_acoustic_subtype"] == "Label-only-TB").sum()
print(f"  Acoustic-TB:    {n_acoustic} ({n_acoustic/len(df_tb)*100:.1f}%)")
print(f"  Label-only-TB:  {n_labelonly} ({n_labelonly/len(df_tb)*100:.1f}%)\n")

print(f"  Acoustic-TB:    {df_tb[df_tb["tb_acoustic_subtype"] == "Acoustic-TB"]['acoustic_tb_score'].mean()}")
print(f"  Label-only-TB:  {df_tb[df_tb["tb_acoustic_subtype"] == "Label-only-TB"]['acoustic_tb_score'].mean()}")

In [ ]:
# Right: Acoustic TB Score distribution
fig, axes = plt.subplots(1, 1, figsize=(14, 6))
ax = axes

tb_idx = np.where(tb_mask)[0]
nontb_idx = np.where(~tb_mask)[0]

nontb_scores = acoustic_scores[~tb_mask]
tb_acoustic_scores = acoustic_scores[tb_mask]

# Split TB by subtype
subtypes = df_tb["tb_acoustic_subtype"].values
acoustic_tb_scores = acoustic_scores[tb_idx[subtypes == "Acoustic-TB"]]
labelonly_tb_scores = acoustic_scores[tb_idx[subtypes == "Label-only-TB"]]

bins = np.linspace(0, 1, 30)
ax.hist(nontb_scores, bins=bins, color="#2980B9", alpha=0.6, label="Non-TB", density=True)
ax.hist(labelonly_tb_scores, bins=bins, color="#F39C12", alpha=0.7, label="Label-only-TB", density=True)
ax.hist(acoustic_tb_scores, bins=bins, color="#C0392B", alpha=0.7, label="Acoustic-TB", density=True)

ax.axvline(threshold, color="black", linestyle="--", linewidth=1.5, label=f"Threshold ({threshold:.2f})")
ax.set_xlabel("Acoustic TB Likelihood Score", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Score Distribution by Subtype", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)

In [ ]:
df_combine["acoustic_tb_score"] = acoustic_scores
subtype_col = np.array([0] * len(df_combine), dtype=object)
tb_idx_positions = np.where(tb_mask)[0]
for i, pos in enumerate(tb_idx_positions):
    subtype_col[pos] = 1 if df_tb["tb_acoustic_subtype"].iloc[i] == "Acoustic-TB" else 2
df_combine["disease_status_rev"] = subtype_col

## Evaluate

In [ ]:
df_filtered = df_combine #[df_combine["disease_status_rev"] != 2].copy().reset_index(drop=True)
df_filtered = df_filtered[df_filtered['db'] == 0]
X = np.vstack(df_filtered["embed"].values).astype(np.float32)
y = df_filtered["disease_status"].astype(int).values
participant = df_filtered["participant"].values
db_labels = df_filtered["db"].values

unique_dbs = np.unique(db_labels)
print(f"\n  Class balance: Acoustic-TB={y.sum()}  Non-TB={(y==0).sum()}")
print(f"  Databases: {dict(zip(*np.unique(db_labels, return_counts=True)))}")
print(f"  Embedding dim: {X.shape[1]}")

In [ ]:
print(f"Scenario A")
classifiers = get_classifiers()
results = {}

cv = GroupKFold(n_splits=5)
splits = list(cv.split(X, y, groups=participant))

for clf_name, clf in classifiers.items():
    print(f"\n  [{clf_name}]")
    fold_aurocs, fold_auprcs = [], []
    all_y_true, all_y_score = [], []
    fold_roc_curves = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_score = clf.predict_proba(X_test)[:, 1]

        auroc = roc_auc_score(y_test, y_score)
        auprc = average_precision_score(y_test, y_score)
        fpr, tpr, _ = roc_curve(y_test, y_score)

        fold_aurocs.append(auroc)
        fold_auprcs.append(auprc)
        all_y_true.extend(y_test)
        all_y_score.extend(y_score)
        fold_roc_curves.append((fpr, tpr))

        print(f"    Fold {fold_idx+1}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}  "
                f"(test n={len(y_test)}, TB={y_test.sum()})")

    mean_auroc = np.mean(fold_aurocs)
    std_auroc  = np.std(fold_aurocs)
    mean_auprc = np.mean(fold_auprcs)

    print(f"    → Mean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f"    → Mean AUPRC: {mean_auprc:.4f}")

    results[clf_name] = {
        "fold_aurocs":      fold_aurocs,
        "mean_auroc":       mean_auroc,
        "std_auroc":        std_auroc,
        "fold_auprcs":      fold_auprcs,
        "mean_auprc":       mean_auprc,
        "all_y_true":       np.array(all_y_true),
        "all_y_score":      np.array(all_y_score),
        "fold_roc_curves":  fold_roc_curves,
    }

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# CLASSIFIERS
# ─────────────────────────────────────────────

def get_classifiers() -> dict:
    """
    Return a dict of classifiers to compare.
    All wrapped in a Pipeline with StandardScaler + PCA for fair comparison.
    PCA(50) reduces 768-dim embeddings to a stable space for all classifiers.
    """
    pca_step = ("pca", PCA(n_components=50, random_state=42))
    scale_step = ("scaler", StandardScaler())

    return {
        "Logistic Regression": Pipeline([
            scale_step, pca_step,
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]),
        "SVM (RBF)": Pipeline([
            scale_step, pca_step,
            ("clf", SVC(kernel="rbf", probability=True, C=1.0, random_state=42)),
        ]),
        "Random Forest": Pipeline([
            scale_step, pca_step,
            ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ]),
        "Gradient Boosting": Pipeline([
            scale_step, pca_step,
            ("clf", GradientBoostingClassifier(n_estimators=200, random_state=42)),
        ]),
    }



# ─────────────────────────────────────────────
# SCENARIO A: 5-Fold Cross-Validation
# ─────────────────────────────────────────────

def run_kfold_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray = None,
    n_splits: int = 5,
    random_state: int = 42,
) -> dict:
    """
    5-Fold Stratified Cross-Validation.

    If groups (participant IDs) provided, uses GroupKFold to ensure
    the same participant never appears in both train and test.
    This is critical to avoid data leakage for repeated cough recordings.

    Returns
    -------
    results : dict[classifier_name] -> {
        'fold_aurocs'     : list of per-fold AUROC
        'mean_auroc'      : float
        'std_auroc'       : float
        'fold_auprcs'     : list of per-fold AUPRC
        'mean_auprc'      : float
        'all_y_true'      : concatenated true labels
        'all_y_score'     : concatenated predicted scores (for global ROC)
        'fold_roc_curves' : list of (fpr, tpr) per fold
    }
    """
    classifiers = get_classifiers()
    results = {}

    if groups is not None:
        # GroupKFold: participant-aware, no stratification guarantee
        # but we shuffle participant order for balance
        cv = GroupKFold(n_splits=n_splits)
        splits = list(cv.split(X, y, groups=groups))
        print(f"  Using GroupKFold (participant-aware) — {n_splits} folds")
    else:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(cv.split(X, y))
        print(f"  Using StratifiedKFold — {n_splits} folds")

    for clf_name, clf in classifiers.items():
        print(f"\n  [{clf_name}]")
        fold_aurocs, fold_auprcs = [], []
        all_y_true, all_y_score = [], []
        fold_roc_curves = []

        for fold_idx, (train_idx, test_idx) in enumerate(splits):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            clf.fit(X_train, y_train)
            y_score = clf.predict_proba(X_test)[:, 1]

            auroc = roc_auc_score(y_test, y_score)
            auprc = average_precision_score(y_test, y_score)
            fpr, tpr, _ = roc_curve(y_test, y_score)

            fold_aurocs.append(auroc)
            fold_auprcs.append(auprc)
            all_y_true.extend(y_test)
            all_y_score.extend(y_score)
            fold_roc_curves.append((fpr, tpr))

            print(f"    Fold {fold_idx+1}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}  "
                  f"(test n={len(y_test)}, TB={y_test.sum()})")

        mean_auroc = np.mean(fold_aurocs)
        std_auroc  = np.std(fold_aurocs)
        mean_auprc = np.mean(fold_auprcs)

        print(f"    → Mean AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
        print(f"    → Mean AUPRC: {mean_auprc:.4f}")

        results[clf_name] = {
            "fold_aurocs":      fold_aurocs,
            "mean_auroc":       mean_auroc,
            "std_auroc":        std_auroc,
            "fold_auprcs":      fold_auprcs,
            "mean_auprc":       mean_auprc,
            "all_y_true":       np.array(all_y_true),
            "all_y_score":      np.array(all_y_score),
            "fold_roc_curves":  fold_roc_curves,
        }

    return results


# ─────────────────────────────────────────────
# SCENARIO B: Cross-Dataset Generalization
# ─────────────────────────────────────────────

def run_cross_dataset(
    X: np.ndarray,
    y: np.ndarray,
    db_labels: np.ndarray,
) -> dict:
    """
    Leave-one-dataset-out evaluation.
    For every pair of databases (A, B):
      - Train on A → Test on B
      - Train on B → Test on A

    This is the strictest generalization test:
    the model never sees ANY sample from the test database during training.
    Directly measures robustness to recording setup / population shift.

    Returns
    -------
    results : dict["{train_db}→{test_db}"] -> {
        classifier_name -> {auroc, auprc, fpr, tpr, y_true, y_score}
    }
    """
    classifiers = get_classifiers()
    unique_dbs = np.unique(db_labels)
    results = {}

    # Generate all directed pairs
    pairs = []
    for i, db_a in enumerate(unique_dbs):
        for db_b in unique_dbs:
            if db_a != db_b:
                pairs.append((db_a, db_b))

    for train_db, test_db in pairs:
        key = f"{train_db} → {test_db}"
        print(f"\n  [{key}]")

        train_mask = db_labels == train_db
        test_mask  = db_labels == test_db

        X_train, y_train = X[train_mask], y[train_mask]
        X_test,  y_test  = X[test_mask],  y[test_mask]

        # Skip if either split has only one class
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"    ⚠ Skipping: single class in train or test split")
            continue

        print(f"    Train: {train_db} (n={len(y_train)}, TB={y_train.sum()}) | "
              f"Test: {test_db} (n={len(y_test)}, TB={y_test.sum()})")

        results[key] = {}
        for clf_name, clf in classifiers.items():
            clf.fit(X_train, y_train)
            y_score = clf.predict_proba(X_test)[:, 1]

            auroc = roc_auc_score(y_test, y_score)
            auprc = average_precision_score(y_test, y_score)
            fpr, tpr, _ = roc_curve(y_test, y_score)

            print(f"    {clf_name:<22}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}")

            results[key][clf_name] = {
                "auroc":   auroc,
                "auprc":   auprc,
                "fpr":     fpr,
                "tpr":     tpr,
                "y_true":  y_test,
                "y_score": y_score,
            }

    return results



In [ ]:
print("=" * 60)
print("STEP 4: CLASSIFICATION & EVALUATION")
print("=" * 60)


In [ ]:
print(f"\n[1/5] Preparing data...")
print(f"  Full df: {len(df)} samples")
print(f"  Subtype distribution:\n  {df_annotated["tb_acoustic_subtype"].value_counts().to_dict()}")

In [ ]:
# Remove Label-only-TB
df_filtered = df_annotated[df_annotated["tb_acoustic_subtype"] != "Label-only-TB"].copy().reset_index(drop=True)
print(f"\n  After removing Label-only-TB: {len(df_filtered)} samples")
print(f"  {df_filtered["tb_acoustic_subtype"].value_counts().to_dict()}")

In [ ]:
# Binary labels
y = (df_filtered["tb_acoustic_subtype"] == "Acoustic-TB").astype(int).values
X = np.vstack(df_filtered["embed_combat"].values).astype(np.float32)
db_labels = df_filtered["db"].values

# Groups for GroupKFold
groups = None
le = LabelEncoder()
groups = le.fit_transform(df_filtered["participant"].values)
print(f"  Unique participants: {len(np.unique(groups))}")

In [ ]:
unique_dbs = np.unique(db_labels)
print(f"\n  Class balance: Acoustic-TB={y.sum()}  Non-TB={(y==0).sum()}")
print(f"  Databases: {dict(zip(*np.unique(db_labels, return_counts=True)))}")
print(f"  Embedding dim: {X.shape[1]}")

In [ ]:
# ── Scenario A: 5-Fold CV ─────────────────────────────────
print(f"\n[2/5] Scenario A: 5-Fold Cross-Validation...")
kfold_results = run_kfold_cv(
    X, y, groups=groups, n_splits=5, random_state=42
)

In [ ]:
# ── Scenario B: Cross-Dataset ─────────────────────────────
print(f"\n[3/5] Scenario B: Cross-Dataset Generalization...")
if len(unique_dbs) < 2:
    print("  ⚠ Only one database found — skipping cross-dataset evaluation")
    cross_results = {}
else:
    cross_results = run_cross_dataset(X, y, db_labels)

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────

COLORS = ["#C0392B", "#2980B9", "#27AE60", "#8E44AD"]

def plot_kfold_roc(kfold_results: dict, save_path: str = None):
    """
    Publication-ready ROC curves for 5-fold CV.
    One panel per classifier, all folds shown as thin lines,
    mean ROC as bold line with AUROC ± std in legend.
    """
    clf_names = list(kfold_results.keys())
    n = len(clf_names)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    fig.suptitle("5-Fold CV — ROC Curves (Acoustic-TB vs Non-TB)",
                 fontsize=14, fontweight="bold")

    for ax, (clf_name, res), color in zip(axes, kfold_results.items(), COLORS):
        # Individual fold curves
        mean_fpr = np.linspace(0, 1, 200)
        interp_tprs = []
        for fold_idx, (fpr, tpr) in enumerate(res["fold_roc_curves"]):
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tprs.append(interp_tpr)
            ax.plot(fpr, tpr, color=color, alpha=0.2, linewidth=1)

        # Mean ROC
        mean_tpr = np.mean(interp_tprs, axis=0)
        std_tpr  = np.std(interp_tprs, axis=0)
        ax.plot(mean_fpr, mean_tpr, color=color, linewidth=2.5,
                label=f"Mean AUROC = {res['mean_auroc']:.3f} ± {res['std_auroc']:.3f}")
        ax.fill_between(mean_fpr,
                        np.clip(mean_tpr - std_tpr, 0, 1),
                        np.clip(mean_tpr + std_tpr, 0, 1),
                        color=color, alpha=0.12, label="±1 SD")

        ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
        ax.set_xlabel("False Positive Rate", fontsize=10)
        ax.set_ylabel("True Positive Rate", fontsize=10)
        ax.set_title(clf_name, fontsize=11, fontweight="bold")
        ax.legend(fontsize=9, loc="lower right")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        _style_ax(ax)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.close()


def plot_kfold_summary(kfold_results: dict, save_path: str = None):
    """
    Bar chart: Mean AUROC ± Std per classifier.
    Includes individual fold dots for transparency.
    """
    clf_names = list(kfold_results.keys())
    means  = [kfold_results[c]["mean_auroc"] for c in clf_names]
    stds   = [kfold_results[c]["std_auroc"]  for c in clf_names]
    auprcs = [kfold_results[c]["mean_auprc"] for c in clf_names]

    x = np.arange(len(clf_names))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("5-Fold CV Summary — Acoustic-TB vs Non-TB",
                 fontsize=13, fontweight="bold")

    for ax, metric_name, metric_vals, metric_stds in [
        (axes[0], "AUROC", means, stds),
        (axes[1], "AUPRC", auprcs, [0]*len(auprcs)),
    ]:
        bars = ax.bar(x, metric_vals, yerr=metric_stds if metric_name == "AUROC" else None,
                      capsize=5, color=COLORS[:len(clf_names)],
                      alpha=0.8, edgecolor="white", linewidth=1.5,
                      error_kw={"linewidth": 2, "ecolor": "black"})

        # Individual fold dots for AUROC
        if metric_name == "AUROC":
            for i, clf_name in enumerate(clf_names):
                folds = kfold_results[clf_name]["fold_aurocs"]
                ax.scatter([i] * len(folds), folds,
                           color="black", zorder=5, s=30, alpha=0.7)

        # Value labels
        for bar, val, std in zip(bars, metric_vals, metric_stds):
            label = f"{val:.3f}" if metric_name == "AUPRC" else f"{val:.3f}±{std:.3f}"
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + (std if metric_name == "AUROC" else 0) + 0.005,
                    label, ha="center", va="bottom", fontsize=9, fontweight="bold")

        ax.set_xticks(x)
        ax.set_xticklabels([c.replace(" ", "\n") for c in clf_names], fontsize=9)
        ax.set_ylabel(metric_name, fontsize=11)
        ax.set_title(f"Mean {metric_name} (5-Fold CV)", fontsize=11, fontweight="bold")
        ax.set_ylim(0, min(1.15, max(metric_vals) * 1.2))
        ax.axhline(0.5, color="gray", linestyle=":", linewidth=1, alpha=0.6, label="Random (0.5)")
        ax.legend(fontsize=9)
        _style_ax(ax)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.close()


def plot_cross_dataset_roc(cross_results: dict, save_path: str = None):
    """
    ROC curves for all cross-dataset directions, one panel per direction.
    All classifiers shown per panel.
    """
    directions = list(cross_results.keys())
    n = len(directions)
    if n == 0:
        return

    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    fig.suptitle("Cross-Dataset Generalization — ROC Curves",
                 fontsize=14, fontweight="bold")

    for ax, direction in zip(axes, directions):
        clf_results = cross_results[direction]
        for (clf_name, res), color in zip(clf_results.items(), COLORS):
            ax.plot(res["fpr"], res["tpr"], color=color, linewidth=2,
                    label=f"{clf_name} (AUROC={res['auroc']:.3f})")

        ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
        ax.set_title(f"Train → Test:\n{direction}", fontsize=11, fontweight="bold")
        ax.set_xlabel("False Positive Rate", fontsize=10)
        ax.set_ylabel("True Positive Rate", fontsize=10)
        ax.legend(fontsize=8, loc="lower right")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        _style_ax(ax)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.close()


def plot_cross_dataset_heatmap(cross_results: dict, save_path: str = None):
    """
    Heatmap: rows = classifiers, columns = train→test directions.
    Color = AUROC. Lets you see at a glance which direction is harder.
    """
    if not cross_results:
        return

    directions = list(cross_results.keys())
    clf_names  = list(next(iter(cross_results.values())).keys())

    matrix = np.zeros((len(clf_names), len(directions)))
    for j, direction in enumerate(directions):
        for i, clf_name in enumerate(clf_names):
            matrix[i, j] = cross_results[direction].get(clf_name, {}).get("auroc", 0)

    fig, ax = plt.subplots(figsize=(max(6, len(directions)*2.5), max(4, len(clf_names)*1.2)))
    sns.heatmap(matrix, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0.5, vmax=1.0,
                xticklabels=directions, yticklabels=clf_names,
                linewidths=0.5, ax=ax,
                cbar_kws={"label": "AUROC", "shrink": 0.8})
    ax.set_title("Cross-Dataset AUROC Heatmap\n(rows=classifier, cols=train→test)",
                 fontsize=12, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.close()


def plot_scenario_comparison(
    kfold_results: dict,
    cross_results: dict,
    save_path: str = None,
):
    """
    Single summary figure comparing 5-fold CV vs cross-dataset AUROC per classifier.
    The gap between CV and cross-dataset is the generalization gap — key for your paper.
    """
    clf_names = list(kfold_results.keys())
    x = np.arange(len(clf_names))
    width = 0.25

    # Collect cross-dataset AUROCs per classifier (mean across directions)
    cross_means = {}
    cross_stds  = {}
    for clf_name in clf_names:
        aurocs = [
            cross_results[direction][clf_name]["auroc"]
            for direction in cross_results
            if clf_name in cross_results[direction]
        ]
        cross_means[clf_name] = np.mean(aurocs) if aurocs else 0
        cross_stds[clf_name]  = np.std(aurocs)  if aurocs else 0

    fig, ax = plt.subplots(figsize=(11, 6))
    fig.suptitle("5-Fold CV vs Cross-Dataset Generalization",
                 fontsize=13, fontweight="bold")

    # 5-fold bars
    bars_cv = ax.bar(
        x - width/2,
        [kfold_results[c]["mean_auroc"] for c in clf_names],
        width, yerr=[kfold_results[c]["std_auroc"] for c in clf_names],
        capsize=5, label="5-Fold CV", color="#2980B9",
        alpha=0.85, edgecolor="white",
        error_kw={"linewidth": 2, "ecolor": "black"}
    )

    # Cross-dataset bars
    bars_cd = ax.bar(
        x + width/2,
        [cross_means[c] for c in clf_names],
        width, yerr=[cross_stds[c] for c in clf_names],
        capsize=5, label="Cross-Dataset (mean)", color="#E74C3C",
        alpha=0.85, edgecolor="white",
        error_kw={"linewidth": 2, "ecolor": "black"}
    )

    # Generalization gap annotation
    for i, clf_name in enumerate(clf_names):
        cv_val = kfold_results[clf_name]["mean_auroc"]
        cd_val = cross_means[clf_name]
        gap    = cv_val - cd_val
        ymax   = max(cv_val, cd_val) + 0.05
        ax.annotate(f"Δ={gap:.3f}", xy=(i, ymax),
                    ha="center", fontsize=8.5, color="#7F8C8D", fontstyle="italic")

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace(" ", "\n") for c in clf_names], fontsize=10)
    ax.set_ylabel("AUROC", fontsize=11)
    ax.set_ylim(0, 1.18)
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1, alpha=0.5)
    ax.legend(fontsize=10)
    _style_ax(ax)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.close()


def _style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.15, linestyle="--")



In [ ]:
print(f"\n[5/5] Generating publication plots...")

plot_kfold_roc(
    kfold_results,
    save_path=f"step4_kfold_roc.png"
)
plot_kfold_summary(
    kfold_results,
    save_path=f"step4_kfold_summary.png"
)

if cross_results:
    plot_cross_dataset_roc(
        cross_results,
        save_path=f"step4_cross_dataset_roc.png"
    )
    plot_cross_dataset_heatmap(
        cross_results,
        save_path=f"step4_cross_dataset_heatmap.png"
    )
    plot_scenario_comparison(
        kfold_results, cross_results,
        save_path=f"step4_scenario_comparison.png"
    )

In [ ]:
# ── STEP 4: Classification ────────────────
    step4_dir = f"{output_dir}/step4_classification"
    step4 = run_classification(
        df=step3["df_annotated"],
        embed_col=best_embed_col,
        disease_col=disease_col,
        db_col=db_col,
        participant_col=participant_col,
        subtype_col="tb_acoustic_subtype",
        n_splits=5,
        output_dir=step4_dir,
        random_seed=random_seed,
    )

In [ ]:
X.shape

In [ ]:
def remove_dataset_bias(row):
    return row.embed - dataset_centroids[row.db]

dataset_centroids = df_combine.groupby("db")["embed"].apply(lambda x: np.mean(np.stack(x), axis=0))
df_combine["embed_debiased"] = df_combine.apply(remove_dataset_bias, axis=1)

X = np.stack(df_combine["embed_debiased"].values)
mu = X.mean(axis=0)
sigma = X.std(axis=0) + 1e-8
X_norm = (X - mu) / sigma

df_combine["embed_debiased"] = list(X_norm)

In [ ]:
# patient_df = (
#     df_combine
#     .groupby("participant")
#     .agg({
#         "embed": lambda x: np.mean(np.stack(x), axis=0),
#         "disease_status": "first",
#         "db": "first",
#     })
#     .reset_index()
# )

patient_df = (
    df_combine
    .groupby("participant")
    .agg({
        "embed": lambda x: np.concatenate([
        np.mean(np.stack(x), axis=0),
        np.std(np.stack(x), axis=0)
        ]),
        "disease_status": "first",
        "db": "first",
    })
    .reset_index()
)

In [ ]:
X_patient_coda = np.stack(patient_df[patient_df['db'] == 0]["embed"].values)
X_patient_cirdz = np.stack(patient_df[patient_df['db'] == 1]["embed"].values)

y_patient_coda = patient_df[patient_df['db'] == 0]["disease_status"].values
y_patient_cirdz = patient_df[patient_df['db'] == 1]["disease_status"].values

# X_patient = np.stack(patient_df["embed"].values)
# y_patient = patient_df["disease_status"].values
# patient_ids = patient_df["participant"].values

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X = np.stack(patient_df["embed"].values)
y = patient_df["db"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

prob = clf.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, prob)
auc

In [ ]:
z = X.copy()

for d in np.unique(y):
    idx = y == d
    mu = z[idx].mean(axis=0)
    std = z[idx].std(axis=0) + 1e-6
    z[idx] = (z[idx] - mu) / std

from sklearn.decomposition import PCA
pca = PCA(n_components=40)
z = pca.fit_transform(z)

X_train, X_test, y_train, y_test = train_test_split(z, y, test_size=0.3, stratify=y)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

prob = clf.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, prob)
auc

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=40,
    min_samples=5
)

clusters = clusterer.fit_predict(z)

y_tb = patient_df["disease_status"].values
global_tb_rate = y_tb.mean()

cluster_stats = {}
for c in np.unique(clusters):
    if c == -1:
        continue

    idx = clusters == c
    tb_rate = y_tb[idx].mean()
    enrichment = tb_rate / global_tb_rate

    cluster_stats[c] = {
        "size": idx.sum(),
        "tb_rate": tb_rate,
        "enrichment": enrichment
    }

print(global_tb_rate)
cluster_stats

In [ ]:
mask_1 = y_patient == 1
all_embeds_train = X_patient[mask_1]
all_labels_train = y_patient[mask_1]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)
X_2d = tsne.fit_transform(X_patient)

plt.figure(figsize=(6, 6))
plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    s=5,
    c=y_patient,
    alpha=0.7
)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE Embedding")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

In [ ]:
mask_1 = y_patient_coda == 1
pos_embeds_train = X_patient_coda[mask_1]

scaler = StandardScaler()
pos_embeds_train_norm = scaler.fit_transform(pos_embeds_train)

gmm = GaussianMixture(
    n_components=6,
    covariance_type="diag",     # key change
    reg_covar=1e-4,             # key change
    n_init=20,
    max_iter=500,
    random_state=42
)
gmm.fit(pos_embeds_train_norm)
tb_clusters = gmm.predict(pos_embeds_train_norm)
print(Counter(tb_clusters))

mask_1 = y_patient_coda == 1
pos_embeds_train = X_patient_coda[mask_1]
neg_embeds_train = X_patient_coda[~mask_1]

aucs = {}
for c in np.unique(tb_clusters):
    if c == -1:
        continue

    pos = pos_embeds_train[tb_clusters == c]
    neg = neg_embeds_train

    X = np.vstack([pos, neg])
    y = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y
    )

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    prob = clf.predict_proba(X_test)[:,1]
    auc_coda = roc_auc_score(y_test, prob)

    X_others = scaler.transform(X_patient_cirdz)
    prob = clf.predict_proba(X_others)[:,1]
    auc_cirdz = roc_auc_score(y_patient_cirdz, prob)

    aucs[c] = [auc_coda, auc_cirdz]

aucs

In [ ]:
aucs

In [ ]:
aucs

In [ ]:


mask_1 = test_labels == 1
pos_embeds_test = test_embeds[mask_1]
pos_labels_test = test_labels[mask_1]
pos_wavs_test = test_wavs[mask_1]

embeds_norm_test = scaler.transform(pos_embeds_test)
test_cluster_ids = gmm.predict(embeds_norm_test)
#cluster_probs = gmm.predict_proba(embeds_norm)

In [ ]:
df_new = pd.DataFrame({
    "path_file": pos_wavs_train,
    "disease_status": train_cluster_ids + 1
})

df_clustered = df_train.merge(
    df_new[['path_file', 'disease_status']],
    on='path_file',
    how='left',
    suffixes=('_old', '_new')
)
df_clustered['disease_status'] = df_clustered['disease_status_new'].fillna(df_clustered['disease_status_old'])
df_clustered = df_clustered[['path_file', 'disease_status', 'sex', 'participant', 'db']]
df_clustered.to_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/clustred_{hps.data.metadata_csv}.train', index=False)

In [ ]:
df_new = pd.DataFrame({
    "path_file": pos_wavs_test,
    "disease_status": test_cluster_ids + 1
})

df_clustered = df_test.merge(
    df_new[['path_file', 'disease_status']],
    on='path_file',
    how='left',
    suffixes=('_old', '_new')
)
df_clustered['disease_status'] = df_clustered['disease_status_new'].fillna(df_clustered['disease_status_old'])
df_clustered = df_clustered[['path_file', 'disease_status', 'sex', 'participant', 'db']]
df_clustered.to_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/clustred_{hps.data.metadata_csv}.test', index=False)

In [ ]:
df_clustered['disease_status'].value_counts()

In [ ]:
mask_0 = train_labels == 0
mask_1 = train_labels == 1

reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.1,
    n_components=2,
    random_state=80
)
emb_2d = reducer.fit_transform(train_embeds)

plt.figure(figsize=(7, 6))
plt.scatter(emb_2d[mask_0, 0], emb_2d[mask_0, 1], s=10, alpha=0.6, label="Label 0")
plt.scatter(emb_2d[mask_1, 0], emb_2d[mask_1, 1], s=10, alpha=0.6, label="Label 1")
plt.legend()
plt.title("Phase-1 Embedding Space (Label 0 vs 1)")
plt.xlabel("Dim-1")
plt.ylabel("Dim-2")
plt.tight_layout()
plt.show()

In [ ]:
len(train_embeds)

In [ ]:
mask_1

In [ ]:
wav_transform = lambda wav: torch.tensor(
                    np.log(librosa.feature.melspectrogram(
                        y=wav.numpy() if isinstance(wav, torch.Tensor) else wav,
                        sr=hps.data.sampling_rate,
                        n_mels=hps.data.n_mel_channels,
                        n_fft=hps.data.filter_length,
                        hop_length=hps.data.hop_length,
                        win_length=hps.data.win_length,
                    ) + 1e-6),
                    dtype=torch.float32,
                )

In [ ]:
input_mel = wav_transform(audio[0].cpu())

fig, axes = plt.subplots(1, 2, figsize=(15, 10))
axes[0].imshow(input_mel[0].cpu().numpy(), aspect='auto', origin='lower')
axes[0].set_title('Original Mel')
axes[1].imshow(reconstructed_mel[0].cpu().numpy(), aspect='auto', origin='lower')
axes[1].set_title('Reconstructed (Clean)')

In [ ]:
mel_ouput = reconstructed_mel[0].cpu().numpy()
mel_ouput = np.exp(mel_ouput)

stft_mag = librosa.feature.inverse.mel_to_stft(
    mel_ouput,
    sr=16000,
    n_fft=2048,
    power=1.0
)

wav = librosa.griffinlim(
    stft_mag,
    n_iter=60,
    hop_length=200,
    win_length=800
)

Audio(wav, rate=16000)

In [ ]:
mel_ouput = input_mel[0].cpu().numpy()
mel_ouput = np.exp(mel_ouput)

stft_mag = librosa.feature.inverse.mel_to_stft(
    mel_ouput,
    sr=16000,
    n_fft=2048,
    power=1.0
)

wav = librosa.griffinlim(
    stft_mag,
    n_iter=60,
    hop_length=200,
    win_length=800
)

Audio(wav, rate=16000)